In [1]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 1.3 MB/s eta 0:00:00


In [3]:
from google.colab import files
uploaded = files.upload()

Saving 12208078_1080_1920_60fps.mp4 to 12208078_1080_1920_60fps.mp4


In [9]:


from ultralytics import YOLO
import cv2

model = YOLO("yolov8n.pt")

input_path = "12208078_1080_1920_60fps.mp4"      # replace with your uploaded file
output_path = "output_professional.mp4"

cap = cv2.VideoCapture(input_path)
fps = int(cap.get(cv2.CAP_PROP_FPS))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

# unique color per class for a cleaner, more "product" look
import random
random.seed(42)
colors = {}

def get_color(cls_id):
    if cls_id not in colors:
        colors[cls_id] = tuple(random.randint(80, 255) for _ in range(3))
    return colors[cls_id]

frame_count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1

    results = model.track(frame, conf=0.4, tracker="bytetrack.yaml", persist=True, verbose=False)
    result = results[0]

    active_ids = set()

    if result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.cpu().numpy().astype(int)
        classes = result.boxes.cls.cpu().numpy().astype(int)
        names = result.names

        for box, track_id, cls in zip(boxes, ids, classes):
            x1, y1, x2, y2 = map(int, box)
            color = get_color(cls)
            active_ids.add(track_id)

            # rounded-looking thick box
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # filled label background for readability
            label = f"{names[cls]} #{track_id}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            cv2.rectangle(frame, (x1, y1 - th - 10), (x1 + tw + 6, y1), color, -1)
            cv2.putText(frame, label, (x1 + 3, y1 - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2)

    # semi-transparent info bar top-left
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (260, 45), (20, 20, 20), -1)
    frame = cv2.addWeighted(overlay, 0.6, frame, 0.4, 0)
    cv2.putText(frame, f"Objects tracked: {len(active_ids)}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

    out.write(frame)

cap.release()
out.release()
print("Saved:", output_path)

Saved: output_professional.mp4


In [10]:
import subprocess
final_path = "output_final.mp4"
subprocess.run(["ffmpeg", "-y", "-i", output_path, "-vcodec", "libx264", "-acodec", "aac", final_path])

from google.colab import files
files.download(final_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>